# Shadow Residual v60 Multiplier Audit

No file I/O. Each code cell defines its own data and prints a PASS signature.

## Claim 1 — retained orientation carries what scalar projection annihilates

In [1]:

import math
SUPPORT={1:1,5:-1,7:-1,11:1}
def chi12(n): return SUPPORT.get(n%12,0)
rows=[]
for n in range(1,80):
    c=chi12(n)
    if c:
        plus=c*n
        minus=-c*n
        rows.append((n,c,plus,minus,plus+minus))
print(rows[:12])
print('PASS' if all(r[-1]==0 for r in rows) and any(r[2] != 0 for r in rows) else 'FAIL')


[(1, 1, 1, -1, 0), (5, -1, -5, 5, 0), (7, -1, -7, 7, 0), (11, 1, 11, -11, 0), (13, 1, 13, -13, 0), (17, -1, -17, 17, 0), (19, -1, -19, 19, 0), (23, 1, 23, -23, 0), (25, 1, 25, -25, 0), (29, -1, -29, 29, 0), (31, -1, -31, 31, 0), (35, 1, 35, -35, 0)]
PASS


## Claim 2 — coefficient-stripping the positive orientation gives eta

In [2]:

def eta_product_coeffs(max_shift):
    coeff={0:1}
    for j in range(1,max_shift+1):
        nxt=dict(coeff)
        for k,v in coeff.items():
            if k+j <= max_shift:
                nxt[k+j]=nxt.get(k+j,0)-v
        coeff={k:v for k,v in nxt.items() if v}
    return coeff

def eta_positive_coeffs(max_shift):
    out={}
    max_n=int(math.sqrt(24*max_shift+1))+2
    for n in range(1,max_n+1):
        c=chi12(n)
        if c:
            m=(n*n-1)//24
            if 0 <= m <= max_shift:
                out[m]=out.get(m,0)+c
    return out
M=300
prod=eta_product_coeffs(M)
pos=eta_positive_coeffs(M)
mis=[k for k in range(M+1) if prod.get(k,0)!=pos.get(k,0)]
print('mismatch_count', len(mis))
print('PASS' if not mis else 'FAIL')


mismatch_count 0
PASS


## Claim 3 — S/T multipliers close the residue-vector parent

In [3]:

import mpmath as mp
mp.mp.dps=60
MOD=12
def theta_vec(tau,N=80):
    return [sum(mp.e**(2*mp.pi*1j*tau*(mp.mpf((MOD*k+r)**2)/24)) for k in range(-N,N+1)) for r in range(MOD)]
def deriv_vec(tau,N=80):
    return [sum((MOD*k+r)*mp.e**(2*mp.pi*1j*tau*(mp.mpf((MOD*k+r)**2)/24)) for k in range(-N,N+1)) for r in range(MOD)]
tau=mp.mpc(0.37,0.91)
Theta=theta_vec(tau); ThetaS=theta_vec(-1/tau); ThetaT=theta_vec(tau+1)
D=deriv_vec(tau); DS=deriv_vec(-1/tau); DT=deriv_vec(tau+1)
err=[]
for r in range(MOD):
    err.append(abs(ThetaT[r]-mp.e**(mp.pi*1j*r*r/12)*Theta[r]))
    err.append(abs(DT[r]-mp.e**(mp.pi*1j*r*r/12)*D[r]))
    predS=mp.sqrt(-1j*tau)/mp.sqrt(12)*sum(mp.e**(-mp.pi*1j*r*s/6)*Theta[s] for s in range(MOD))
    predDS=1j*(-1j*tau)**(mp.mpf(3)/2)/mp.sqrt(12)*sum(mp.e**(-mp.pi*1j*r*s/6)*D[s] for s in range(MOD))
    err.append(abs(ThetaS[r]-predS))
    err.append(abs(DS[r]-predDS))
print('max_error', max(err))
print('PASS' if max(err) < mp.mpf('1e-40') else 'FAIL')


max_error 1.32067372689053341283984306689039388222327139275940865398438e-60
PASS


## Claim 4 — eta support vector has eta multipliers

In [4]:

mp.mp.dps=60
chi=[SUPPORT.get(r,0) for r in range(12)]
serr=[]
terr=[]
for r in range(12):
    lhs=sum(mp.e**(-mp.pi*1j*r*s/6)*chi[s] for s in range(12))/mp.sqrt(12)
    serr.append(abs(lhs-chi[r]))
    lhsT=mp.e**(mp.pi*1j*r*r/12)*chi[r]
    rhsT=mp.e**(mp.pi*1j/12)*chi[r]
    terr.append(abs(lhsT-rhsT))
print('S_max_error', max(serr))
print('T_max_error', max(terr))
print('PASS' if max(serr+terr) < mp.mpf('1e-40') else 'FAIL')


S_max_error 1.88063513864145299130930000937012565254585407613053822064712e-60
T_max_error 1.34113288554919112308111287357795311468589723837491751319588e-60
PASS


## Result

The scalar obstruction remains correct. The multiplier surface closes on the retained orientation-by-residue channel. The coefficient-stripped shadow is eta.